# Project 2: Time-Series EDA


# Exploratory Data Analysis (EDA)
This notebook follows a reusable EDA workflow: inspect the data, check data quality, summarize distributions, visualize patterns, investigate anomalies, and document findings.

> The dataset is generated inside this notebook so the project runs without downloading files. Replace the sample data with a real CSV later if desired.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
sns.set_theme()

## 2. Create sample daily data

In [ ]:
rng = np.random.default_rng(17)
dates = pd.date_range("2024-01-01", "2025-12-31", freq="D")
t = np.arange(len(dates))
weekly_effect = np.where(pd.Series(dates).dt.dayofweek.to_numpy() >= 5, -18, 8)
annual_effect = 25 * np.sin(2 * np.pi * t / 365.25)
trend = 0.06 * t
noise = rng.normal(0, 12, len(dates))
sales = 120 + trend + annual_effect + weekly_effect + noise
df = pd.DataFrame({"date": dates, "sales": sales.round(2), "website_visits": (900 + 2.1*sales + rng.normal(0, 70, len(dates))).round().astype(int)})
# Add missing dates values and one unusual spike for anomaly investigation
df.loc[[35, 144, 320], "sales"] = np.nan
df.loc[df["date"] == "2025-11-28", "sales"] *= 2.5
df.head()

## 3. Inspect and validate

In [ ]:
print("Shape:", df.shape)
display(df.head())
df.info()
display(df.describe().T)
print("Duplicate dates:", df["date"].duplicated().sum())
print("Missing values:\n", df.isna().sum())

## 4. Prepare time index and missing values

In [ ]:
eda = df.copy()
eda["date"] = pd.to_datetime(eda["date"], errors="coerce")
eda = eda.dropna(subset=["date"]).sort_values("date").set_index("date")
# Time interpolation is shown for practice; choose a method that fits the real business context.
eda["sales"] = eda["sales"].interpolate(method="time")
print("Missing after interpolation:\n", eda.isna().sum())
print("Date range:", eda.index.min(), "to", eda.index.max())
print("Daily frequency gaps:", len(pd.date_range(eda.index.min(), eda.index.max(), freq="D").difference(eda.index)))

## 5. Trend and rolling statistics

In [ ]:
eda["rolling_7d"] = eda["sales"].rolling(7, min_periods=1).mean()
eda["rolling_30d"] = eda["sales"].rolling(30, min_periods=1).mean()
ax = eda[["sales", "rolling_7d", "rolling_30d"]].plot(figsize=(14, 5), alpha=.8)
ax.set_title("Daily Sales and Rolling Averages")
ax.set_ylabel("Sales")
plt.tight_layout()
plt.show()

## 6. Resampling and seasonality

In [ ]:
monthly = eda[["sales", "website_visits"]].resample("MS").agg({"sales":"sum", "website_visits":"mean"})
weekly = eda["sales"].resample("W").sum()
display(monthly.head())
fig, axes = plt.subplots(2, 1, figsize=(13, 8))
monthly["sales"].plot(ax=axes[0], marker="o")
axes[0].set_title("Monthly Total Sales")
eda.groupby(eda.index.dayofweek)["sales"].mean().plot(kind="bar", ax=axes[1])
axes[1].set_title("Average Sales by Day of Week (0=Monday)")
axes[1].set_xlabel("Day of week")
plt.tight_layout()
plt.show()

## 7. Distribution and descriptive statistics

In [ ]:
display(eda[["sales", "website_visits"]].describe().T)
print("Sales variance:", round(eda["sales"].var(), 2))
print("Sales standard deviation:", round(eda["sales"].std(), 2))
print("Sales skewness:", round(eda["sales"].skew(), 3))
print("Sales kurtosis:", round(eda["sales"].kurtosis(), 3))
sns.histplot(eda["sales"], kde=True)
plt.title("Sales Distribution")
plt.show()

## 8. Correlation analysis

In [ ]:
corr = eda[["sales", "website_visits"]].corr()
display(corr)
sns.regplot(data=eda, x="website_visits", y="sales", scatter_kws={"alpha":.25})
plt.title("Website Visits vs Sales")
plt.show()

## 9. Detect potential anomalies using rolling z-score

In [ ]:
rolling_mean = eda["sales"].rolling(30, min_periods=10).mean()
rolling_std = eda["sales"].rolling(30, min_periods=10).std()
eda["rolling_zscore"] = (eda["sales"] - rolling_mean) / rolling_std.replace(0, np.nan)
anomalies = eda.loc[eda["rolling_zscore"].abs() > 3, ["sales", "rolling_mean" if "rolling_mean" in eda else "sales", "rolling_zscore"]]
print("Potential anomalies:", len(anomalies))
display(eda.loc[eda["rolling_zscore"].abs() > 3, ["sales", "rolling_zscore"]].head(20))
# Rolling anomaly flags require context; calendar events can explain genuine spikes.

## 10. Basic statistical comparison

In [ ]:
from scipy.stats import ttest_ind
weekday = eda.loc[eda.index.dayofweek < 5, "sales"]
weekend = eda.loc[eda.index.dayofweek >= 5, "sales"]
stat, p = ttest_ind(weekday, weekend, equal_var=False, nan_policy="omit")
print(f"Weekday vs weekend Welch t-test: statistic={stat:.3f}, p-value={p:.5f}")
print("At alpha=0.05:", "evidence of a difference in average sales" if p < .05 else "insufficient evidence of a difference")
print("Check assumptions and practical significance before acting on this result.")

## 11. Save outputs

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
eda.to_csv(output_dir / "timeseries_eda_results.csv")
monthly.to_csv(output_dir / "monthly_summary.csv")
fig, ax = plt.subplots(figsize=(12, 4))
monthly["sales"].plot(ax=ax, marker="o", title="Monthly Sales")
plt.tight_layout()
plt.savefig(output_dir / "monthly_sales.png", dpi=150)
plt.close(fig)
print("Saved outputs to:", output_dir.resolve())

## Conclusion

Document overall trend, weekday/weekend differences, seasonal hints, missing-value handling, flagged anomalies, and limits. This dataset is synthetic and the patterns are illustrative, not real business evidence.